In [30]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from imblearn.under_sampling import RandomUnderSampler
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.metrics import classification_report
from torch.autograd import detect_anomaly
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, \
    GenerationConfig
import google.generativeai as genai
from google.generativeai import types
import os
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import google.api_core.exceptions
from dotenv import load_dotenv
from openai import OpenAI
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from accelerate import Accelerator

In [31]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SATD Detection Dataset

In [32]:
detect_train_df = pd.read_csv('../data/detect_train.csv')
detect_train_dataset = Dataset.from_pandas(detect_train_df)
detect_test_df = pd.read_csv('../data/detect_test.csv')
detect_test_dataset = Dataset.from_pandas(detect_test_df)
detect_train_balanced_df = pd.read_csv('../data/detect_train_balanced.csv')
detect_train_balanced_dataset = Dataset.from_pandas(detect_train_balanced_df)

# SATD Classification Dataset

In [33]:
classify_train_df = pd.read_csv('../data/classify_train.csv')
classify_train_dataset = Dataset.from_pandas(classify_train_df)
classify_test_df = pd.read_csv('../data/classify_test.csv')
classify_test_dataset = Dataset.from_pandas(classify_test_df)

# N-Shots SATD Detection Dataset

In [34]:
detect_n_shot_df = pd.read_csv('../data/detect_n_shot.csv')
detect_n_shot_dataset = Dataset.from_pandas(detect_n_shot_df)
detect_n_shot_dataset

Dataset({
    features: ['id', 'repository', 'text', 'label', 'code_before', 'code_after', 'cot'],
    num_rows: 15
})

In [35]:
from util import get_first_n_line, get_last_n_line
from jinja2 import Template


class PromptTemplate:
    def __init__(self, name, definition, instruction, n_shot_template, line_m_before, line_n_after):
        self._name = name
        self._definition = definition
        self._instruction = instruction
        self._n_shot_template = n_shot_template
        self._line_m_before = line_m_before
        self._line_n_after = line_n_after

    @property
    def name(self):
        return self._name

    @property
    def definition(self):
        return self._definition

    @property
    def instruction(self):
        return self._instruction

    @property
    def line_m_before(self):
        return self._line_m_before

    @property
    def line_n_after(self):
        return self._line_n_after

    @property
    def shot_template(self):
        return self._n_shot_template

    def create_example(self, args):
        properties = dict(args)
        if 'code_before' in properties:
            properties['code_before'] = get_last_n_line(args['code_before'], self.line_m_before)
        if 'code_after' in properties:
            properties['code_after'] = get_first_n_line(args['code_after'], self.line_n_after)

        return Template(self.shot_template).render(**properties)

    def create_prompt(self, examples: [str]):
        return self.definition + "\n" + self.instruction + "\n" + "\n" + "\n\n".join(examples)

    def __repr__(self):
        return f"PromptTemplate(name={self.name}, description='{self.definition}', example='{self.shot_template}')"

In [36]:
from enum import Enum


class TrainStrategy(Enum):
    N_SHOT_RANDOM = 'n_shot_random'
    N_SHOT_SIMILAR = 'n_shot_similar'
    N_SHOT_TOP = 'n_shot_top'
    ALL = 'all'


In [37]:
import random
from sentence_transformers import SentenceTransformer, util

sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')


def pick_n_shot(train_dataset: Dataset, test_dataset: Dataset, test_index: int, n: int = 0,
                strategy: TrainStrategy = None):
    dataset_length = train_dataset.num_rows
    if dataset_length < n:
        raise Exception(f'Train dataset contains only {dataset_length} examples for {n} shots')
    indexes = []
    if strategy == TrainStrategy.N_SHOT_RANDOM:
        indexes = random.sample(dataset_length, n)
    elif strategy == TrainStrategy.N_SHOT_SIMILAR:
        similarities = util.cos_sim(sentence_transformer.encode(test_dataset['text'][test_index]),
                                    sentence_transformer.encode(train_dataset['text'])).squeeze(0).numpy()
        top_n_indices = np.argpartition(similarities, -n)[-n:]
        indexes = top_n_indices[np.argsort(similarities[top_n_indices])[::-1]].tolist()
    elif strategy == TrainStrategy.N_SHOT_TOP:
        indexes = [i for i in range(n)]
    return indexes


In [38]:
def report_mismatch(file: str):
    _, name = os.path.basename(file).split('$', 1)
    merged_file = f'{os.path.dirname(file)}/merged_{name}'
    mismatched_file = f'{os.path.dirname(file)}/mismatched_{name}'
    last_df = pd.read_csv(file)

    for f in [merged_file, mismatched_file]:
        if not os.path.exists(merged_file):
            pd.DataFrame(columns=last_df.columns).to_csv(f, index=False)

    merged_df = pd.read_csv(merged_file)
    ids = merged_df['id'].values
    for index, row in last_df.iterrows():
        if row['id'] in ids:
            merged_df.loc[merged_df['id'] == row['id'], 'label_pred'] = row['label_pred']
        else:
            merged_df.loc[len(merged_df)] = row
    merged_df.sort_values(by=['id'], ascending=True, inplace=True)
    merged_df.to_csv(merged_file, index=False)
    merged_df[merged_df['label'] != merged_df['label_pred']].to_csv(mismatched_file, index=False)

In [39]:
def print_classification_excluding_outlier_repository(input_file: str, repository_id: int = 69):
    result_df = pd.read_csv(input_file)
    filtered_result_df = result_df[result_df['repository'] != repository_id]
    print(f'Test Result Excluding repository: {repository_id}')
    print(classification_report(filtered_result_df['label'], filtered_result_df['label_pred'], zero_division=0, digits=3))

In [43]:
from typing import List
from abc import abstractmethod

N_SHOT_PROPERTIES = ['text', 'label', 'code_before', 'code_after', 'cot']


class Model:
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str,
                 verbose: bool = False):
        self.task_type = task_type
        self.model_uri = model_uri
        self.known_labels = set([label.lower() for label in known_labels])
        self.unmatched_label = unmatched_label
        self.verbose = verbose
        self.unknown_labels = []

    @abstractmethod
    def fit(self, dataset: Dataset):
        pass

    @abstractmethod
    def predict(self, dataset: Dataset):
        pass

    def format_label(self, label):
        if label.lower() in self.known_labels:
            return label.lower()
        else:
            if self.verbose:
                print(f'Unknown Label: {label}')
            self.unknown_labels.append(label)
            return self.unmatched_label

    def predict_start(self, dataset: Dataset):
        print(f'{self.task_type} with {self.model_uri.split("/")[-1]}')
        self.unknown_labels.clear()

    def predict_end(self, dataset: Dataset, label_predictions):
        file_name = f'{self.task_type}_{self.model_uri.split("/")[-1]}'
        test_output = dataset.to_dict()
        test_output['label_pred'] = label_predictions
        if self.verbose:
            print(f'Unknown Labels:\n{self.unknown_labels}')
        print('Test Result:')
        print(classification_report(dataset['label'], label_predictions, zero_division=0, digits=3))
        timestamp = datetime.now().strftime("%B %d, %Y, %H:%M:%S")
        file = f'./cache/{timestamp}${file_name}.csv'
        Dataset.from_dict(test_output).to_pandas().to_csv(file, index=False)
        report_mismatch(file)

        return file

    def project_properties(self, dataset: Dataset, index: int):
        properties = {}
        for key in N_SHOT_PROPERTIES:
            if key in dataset.features.keys():
                properties[key] = dataset[key][index]
        return properties

    def create_prompt(self, prompt_template: PromptTemplate, train_dataset: Dataset, train_indexes: [int],
                      test_dataset: Dataset,
                      test_index: int):
        examples = [prompt_template.create_example(self.project_properties(train_dataset, index)) for index in
                    train_indexes]

        test_properties = self.project_properties(test_dataset, test_index)
        if 'label' in test_properties:
            test_properties['label'] = ''
        examples.append(prompt_template.create_example(test_properties))

        prompt = prompt_template.create_prompt(examples)
        if self.verbose:
            print(f'Prompt:\n {prompt}')
        return prompt

In [41]:
@retry(
    stop=stop_after_attempt(10),  # Stop after 5 retries
    wait=wait_exponential(multiplier=2, min=60, max=2 * 60),
    retry=retry_if_exception_type(google.api_core.exceptions.ResourceExhausted),  # Retry on rate limit errors
)
def predict_with_gemini(model, prompt):
    generation_config = types.GenerationConfig(
        temperature=0.0
    )
    return model.generate_content(contents=prompt, generation_config=generation_config).text.split()[-1].lower()


class GeminiModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str,
                 prompt_template: PromptTemplate, train_strategy: TrainStrategy, n_shot_size: int,
                 verbose: bool = False):
        super().__init__(task_type, model_uri, known_labels, unmatched_label, verbose)
        self.model = genai.GenerativeModel(model_uri)
        self.prompt_template = prompt_template
        self.train_strategy = train_strategy
        self.n_shot_size = n_shot_size
        self.train_dataset = None
        self.train_indexes = None
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

    def fit(self, dataset: Dataset):
        self.train_dataset = dataset

    def predict(self, dataset: Dataset):
        super().predict_start(dataset)
        label_predictions = []
        for index in range(dataset.num_rows):
            train_indexes = pick_n_shot(self.train_dataset, dataset, index, self.n_shot_size, self.train_strategy)
            prompt = self.create_prompt(self.prompt_template, self.train_dataset, train_indexes, dataset, index)
            label_predictions.append(self.format_label(predict_with_gemini(self.model, prompt)))
        return super().predict_end(dataset, label_predictions)


# Detect with Gemini 2.0 Flash N-Shots


In [22]:
template = PromptTemplate(
    name="Manually Crafted",
    definition="You are Code Expert trained to detect Self-Admitted Technical Debt (SATD) in Java test code comments. Use this context when making your classification. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional information indicating the need for future improvement. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Classify by labelling 'yes' if the comment include a strong indication of Self-Admitted Technical Debt otherwise label with 'no'.",
    n_shot_template="""
    <EXAMPLE>
    Comment: {{ text }}
    {% if cot -%}
    Reason: {{ cot }}
    {% endif -%}
    Label: {{ label }}
    </EXAMPLE>""",
    line_m_before=3,
    line_n_after=3
)
# when  you will be provided with a comment and, if available, surrounding code for context,  reason as Chain of Thought . Use this context to ensure the comment genuinely indicates SATD, as some may appear to express debt but merely describe the code..

# template = PromptTemplate(
#     name="Manually Crafted",
#     definition="Self-Admitted Technical Debt (SATD) is comment in source code that indicates the need for future improvement or fixes. Developers generally uses  phases like TODO, fixme, etc., as an indication.",
#     instruction="Your job is to detect a given comment as Yes for SATD and No for not-SATD. Please don't predict anything else. Do not comment as Yes if developers don't mention explicitly the need for future improvement. Also, label as No if your confidence is low",
#     n_shot_template="<EXAMPLE>\nComment: {label}\nLabel: {label}\n</EXAMPLE>",
#     line_m_before=3,
#     line_n_after=10
# )
# gemini-2.0-pro-exp-02-05

# test_dataset = Dataset.from_pandas(pd.read_csv('./cache/mismatched_detect_gemini-2.0-flash.csv')[:])
test_dataset = Dataset.from_pandas(detect_test_df[:1])
gemini_model = GeminiModel('detect', 'models/gemini-2.0-flash', {'yes', 'no'}, 'no', template, TrainStrategy.N_SHOT_TOP,
                           10, verbose=False)
gemini_model.fit(detect_n_shot_dataset)
gemini_model.predict(test_dataset)


detect with gemini-2.0-flash
              precision    recall  f1-score   support

          no      0.000     0.000     0.000       1.0
         yes      0.000     0.000     0.000       0.0

    accuracy                          0.000       1.0
   macro avg      0.000     0.000     0.000       1.0
weighted avg      0.000     0.000     0.000       1.0

Merging Mismatch


'./cache/March 25, 2025, 13:52:20$detect_gemini-2.0-flash.csv'

In [12]:
class HuggingFaceModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str,
                 prompt_template: PromptTemplate, train_strategy: TrainStrategy, n_shot_size: int,
                 verbose: bool = False):
        super().__init__(task_type, model_uri, known_labels, unmatched_label, verbose)
        self.tokenizer = AutoTokenizer.from_pretrained(model_uri)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_uri, device_map="auto")
        self.prompt_template = prompt_template
        self.train_strategy = train_strategy
        self.n_shot_size = n_shot_size
        self.train_dataset = None
        self.train_indexes = None

    def fit(self, dataset: Dataset):
        self.train_dataset = dataset

    def predict(self, dataset: Dataset):
        super().predict_start(dataset)
        label_predictions = []
        for index in range(dataset.num_rows):
            train_indexes = pick_n_shot(self.train_dataset, dataset, index, self.n_shot_size, self.train_strategy)
            prompt = self.create_prompt(self.prompt_template, self.train_dataset, train_indexes, dataset, index)
            tokens = self.tokenizer(prompt, return_tensors="pt")
            # tokens['input_ids'] = tokens.input_ids.to(self.model.device)
            input_ids = tokens.input_ids.to(self.model.device)
            # print(len(input_ids[0]))
            # detokenized_text = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
            # print(detokenized_text)
            output = self.model.generate(input_ids)
            # print(self.tokenizer.decode(outputs[0], skip_special_tokens=False))
            label_predictions.append(self.format_label(self.tokenizer.decode(output[0], skip_special_tokens=True)))
        return super().predict_end(dataset, label_predictions)


# Detect with google/flan-t5-base Base N-Shots

In [36]:
template = PromptTemplate(
    name="Manually Crafted",
    definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Classify by labelling 'yes' if the comment include a strong indication of Self-Admitted Technical Debt otherwise label with 'no'.",
    n_shot_template="""
    <EXAMPLE>
    Comment: {{ text }}
    {% if cot -%}
    Reason: {{ cot }}
    {% endif -%}
    Label: {{ label }}
    </EXAMPLE>""",
    line_m_before=3,
    line_n_after=10
)

test_dataset = Dataset.from_pandas(detect_test_df[:])
# test_dataset = Dataset.from_pandas(pd.read_csv('./cache/mismatched_detect_flan-t5-base.csv')[:1])
flan_model = HuggingFaceModel('detect', 'google/flan-t5-base', {'yes', 'no'}, 'no', template,
                              TrainStrategy.N_SHOT_SIMILAR,
                              3, verbose=False)
flan_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_model.predict(test_dataset))


Token indices sequence length is longer than the specified maximum sequence length for this model (1167 > 512). Running this sequence through the model will result in indexing errors


cuda:0
detect with flan-t5-base
              precision    recall  f1-score   support

          no      1.000     1.000     1.000         3

    accuracy                          1.000         3
   macro avg      1.000     1.000     1.000         3
weighted avg      1.000     1.000     1.000         3

Merging Mismatch


'./cache/March 25, 2025, 14:02:42$detect_flan-t5-base.csv'

# Detect with google/flan-t5-large N-Shots

In [ ]:
template = PromptTemplate(
    name="Manually Crafted",
    definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is not final, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Classify by labelling 'yes' if the comment include a strong indication of Self-Admitted Technical Debt otherwise label with 'no'.",
    n_shot_template="""
    <EXAMPLE>
    Comment: {{ text }}
    {% if cot -%}
    Reason: {{ cot }}
    {% endif -%}
    Label: {{ label }}
    </EXAMPLE>""",
    line_m_before=3,
    line_n_after=10
)
test_dataset = Dataset.from_pandas(detect_test_df[:])
# test_dataset = Dataset.from_pandas(pd.read_csv('./cache/mismatched_detect_flan-t5-large.csv')[:])
flan_model = HuggingFaceModel('detect', 'google/flan-t5-large', {'yes', 'no'}, 'no', template, TrainStrategy.N_SHOT_TOP, 3,
                              verbose=False)
flan_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_model.predict(test_dataset))


# Detect with google/flan-t5-xl N-Shots

In [13]:
template = PromptTemplate(
    name="Manually Crafted",
    definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is not final, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Classify by labelling 'yes' if the comment include a strong indication of Self-Admitted Technical Debt otherwise label with 'no'.",
    n_shot_template="""
    <EXAMPLE>
    Comment: {{ text }}
    {% if cot -%}
    Reason: {{ cot }}
    {% endif -%}
    Label: {{ label }}
    </EXAMPLE>""",
    line_m_before=3,
    line_n_after=10
)
test_dataset = Dataset.from_pandas(detect_test_df[:])
# test_dataset = Dataset.from_pandas(pd.read_csv('./cache/mismatched_detect_flan-t5-xl.csv')[:])
flan_model = HuggingFaceModel('detect', 'google/flan-t5-xl', {'yes', 'no'}, 'no', template, TrainStrategy.N_SHOT_TOP, 3,
                              verbose=False)
flan_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_model.predict(test_dataset))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

detect with flan-t5-xl


Token indices sequence length is longer than the specified maximum sequence length for this model (630 > 512). Running this sequence through the model will result in indexing errors


              precision    recall  f1-score   support

          no      0.885     0.451     0.597        51
         yes      0.034     0.250     0.061         4

    accuracy                          0.436        55
   macro avg      0.460     0.350     0.329        55
weighted avg      0.823     0.436     0.558        55

Merging Mismatch


'./cache/March 25, 2025, 15:13:10$detect_flan-t5-xl.csv'

# Detect with google/flan-t5-xxl N-Shots

In [ ]:
template = PromptTemplate(
    name="Manually Crafted",
    definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is not final, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Classify by labelling 'yes' if the comment include a strong indication of Self-Admitted Technical Debt otherwise label with 'no'.",
    n_shot_template="""
    <EXAMPLE>
    Comment: {{ text }}
    {% if cot -%}
    Reason: {{ cot }}
    {% endif -%}
    Label: {{ label }}
    </EXAMPLE>""",
    line_m_before=3,
    line_n_after=10
)
test_dataset = Dataset.from_pandas(detect_test_df[:])
# test_dataset = Dataset.from_pandas(pd.read_csv('./cache/mismatched_detect_flan-t5-xxl.csv')[:])
flan_model = HuggingFaceModel('detect', 'google/flan-t5-xxl', {'yes', 'no'}, 'no', template, TrainStrategy.N_SHOT_TOP, 3,
                              verbose=False)
flan_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_model.predict(test_dataset))


In [100]:
class ChatGpt4Model(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str,
                 prompt_template: PromptTemplate, train_strategy: TrainStrategy, n_shot_size: int,
                 verbose: bool = False):
        super().__init__(task_type, model_uri, known_labels, unmatched_label, verbose)
        self.client = OpenAI(api_key=os.getenv("OPEN_AI_API_KEY"))
        self.prompt_template = prompt_template
        self.train_strategy = train_strategy
        self.n_shot_size = n_shot_size
        self.train_dataset = None
        self.train_indexes = None
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

    def fit(self, dataset: Dataset):
        self.train_dataset = dataset

    def predict(self, dataset: Dataset):
        super().predict_start(dataset)
        label_predictions = []
        for index in range(dataset.num_rows):
            train_indexes = pick_n_shot(self.train_dataset, dataset, index, self.n_shot_size, self.train_strategy)
            prompt = self.create_prompt(self.prompt_template, self.train_dataset, train_indexes, dataset, index)
            completion = self.client.chat.completions.create(
                model=self.model_uri,
                store=True,
                messages=[
                    {"role": "user", "content": prompt}
                ])
            label_pred = completion.choices[0].message.content.strip().split()[-1].lower()
            label_predictions.append(self.format_label(label_pred))
        return super().predict_end(dataset, label_predictions)


# Detect with gpt-4o-mini N-Shots

In [102]:
template = PromptTemplate(
    name="Manually Crafted",
    definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
    n_shot_template="<EXAMPLE>\nComment: {text}\nLabel: {label}\n</EXAMPLE>",
    line_m_before=3,
    line_n_after=10
)

# test_dataset = Dataset.from_pandas(detect_test_dataset)
# test_dataset = Dataset.from_pandas(pd.read_csv('./cache/mismatched_detect_gpt-4o-mini.csv')[:])
test_dataset = Dataset.from_pandas(detect_test_df[:20])
chat_gpt4o_mini_model = ChatGpt4Model('detect', 'gpt-4o-mini', {'yes', 'no'}, 'no', template,
                                      TrainStrategy.N_SHOT_TOP, 3, verbose=False)
chat_gpt4o_mini_model.fit(detect_n_shot_dataset)
chat_gpt4o_mini_model.predict(test_dataset)


detect with gpt-4o-mini
              precision    recall  f1-score   support

          no      1.000     0.600     0.750        20
         yes      0.000     0.000     0.000         0

    accuracy                          0.600        20
   macro avg      0.500     0.300     0.375        20
weighted avg      1.000     0.600     0.750        20

Merging Mismatch


'./cache/March 23, 2025, 04:07:07$detect_gpt-4o-mini.csv'

# Detect with gpt-4o N-Shots

In [103]:
template = PromptTemplate(
    name="Manually Crafted",
    definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
    n_shot_template="<EXAMPLE>\nComment: {text}\nLabel: {label}\n</EXAMPLE>",
    line_m_before=3,
    line_n_after=10
)

# test_dataset = Dataset.from_pandas(detect_test_dataset)
# test_dataset = Dataset.from_pandas(pd.read_csv('./cache/mismatched_detect_gpt-4o.csv')[:])
test_dataset = Dataset.from_pandas(detect_test_df[:20])
chat_gpt4o_model = ChatGpt4Model('detect', 'gpt-4o', {'yes', 'no'}, 'no', template,
                                 TrainStrategy.N_SHOT_TOP, 3, verbose=False)
chat_gpt4o_model.fit(detect_n_shot_dataset)
chat_gpt4o_model.predict(test_dataset)


detect with gpt-4o
              precision    recall  f1-score   support

          no      1.000     0.850     0.919        20
         yes      0.000     0.000     0.000         0

    accuracy                          0.850        20
   macro avg      0.500     0.425     0.459        20
weighted avg      1.000     0.850     0.919        20

Merging Mismatch


'./cache/March 23, 2025, 04:15:01$detect_gpt-4o.csv'

In [21]:
class SentenceEmbeddedLogisticsRegressionModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str,
                 verbose: bool = False):
        super().__init__(task_type, model_uri, known_labels, unmatched_label, verbose)
        self.transformer = SentenceTransformer(model_uri)
        self.model = LogisticRegression()

    def fit(self, dataset: Dataset):
        self.model.fit(self.transformer.encode(dataset['text']), dataset['label'])

    def predict(self, dataset: Dataset):
        super().predict_start(dataset)
        label_predictions = self.model.predict(self.transformer.encode(dataset['text']))
        for index in range(dataset.num_rows):
            label_predictions[index] = self.format_label(label_predictions[index])
        return super().predict_end(dataset, label_predictions)


# Detect with `all-MiniLM-L6-v2` Embedding and Logistic Regression

In [42]:
#Logistic Regression
# under_sampler = RandomUnderSampler(sampling_strategy='auto', random_state=42)
# lr_df_all = train_dataset_all.to_pandas()
# indices, _ = under_sampler.fit_resample(lr_df_all.index.values.reshape(-1, 1), lr_df_all['satd'])
# lr_df_resampled = lr_df_all.loc[indices.flatten()]
# lr_df_resampled.reset_index(drop=True)
# lr_dataset = Dataset.from_pandas(lr_df_resampled)


sentence_embedded_model = SentenceEmbeddedLogisticsRegressionModel('detect', 'all-MiniLM-L6-v2', {'yes', 'no'}, 'no')
sentence_embedded_model.fit(detect_train_balanced_dataset)
print_classification_excluding_outlier_repository(sentence_embedded_model.predict(detect_test_dataset))

detect with all-MiniLM-L6-v2
              precision    recall  f1-score   support

          no      0.997     0.932     0.963      7592
         yes      0.229     0.864     0.363       177

    accuracy                          0.931      7769
   macro avg      0.613     0.898     0.663      7769
weighted avg      0.979     0.931     0.950      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.997     0.932     0.963      7495
         yes      0.134     0.767     0.228       103

    accuracy                          0.929      7598
   macro avg      0.565     0.849     0.595      7598
weighted avg      0.985     0.929     0.953      7598



# Detect with `all-MiniLM-L6-v2` Embedding including Prompt and Logistic Regression

In [49]:
# model_config = ModelConfig(name="Sentence Embedded Logistic Regression", architecture="sentence-embedded-regression",
#                            uri="all-MiniLM-L6-v2")
# template = PromptTemplate(
#     name="Manually Crafted",
#     definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
#     instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
#     n_shot_template="Comment: {comment}\nLabel: {label}",
#     line_m_before=3,
#     line_n_after=10
# )
# train_df = detect_train_balanced_df.copy()
# test_df = detect_test_df.copy()
# for df in [train_df, test_df]:
#     df['text'] = df['text'].apply(lambda text: create_prompt(template, detect_train_dataset,[], test_dataset,
#                                                              ?))
# flan_model = SentenceEmbeddedLogisticsRegressionModel('detect', 'all-MiniLM-L6-v2', {'yes', 'no'}, 'no')
# flan_model.fit(detect_train_dataset)
# flan_model.predict(detect_test_dataset)

Running all-MiniLM-L6-v2
              precision    recall  f1-score   support

          no      0.995     0.931     0.962     12655
         yes      0.213     0.805     0.337       292

    accuracy                          0.928     12947
   macro avg      0.604     0.868     0.649     12947
weighted avg      0.978     0.928     0.948     12947

Unknown classification count 0
Unknown classification  []


In [94]:
import jpype
import jpype.imports
from jpype.types import *
from dotenv import load_dotenv
import os

load_dotenv()


class TextMiningBasedSatdDetectorModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str,
                 verbose: bool = False):
        super().__init__(task_type, model_uri, known_labels, unmatched_label, verbose)

    def fit(self, dataset: Dataset):
        pass

    def predict(self, dataset: Dataset):
        super().predict_start(dataset)
        label_predictions = []

        if not jpype.isJVMStarted():
            jar_path = os.getenv('SATD_DETECTOR_JAR')
            dependency_path = os.getenv('SATD_DETECTOR_DEPENDENCY')
            jvm_args = ["-Xss512m"]
            jpype.startJVM(jpype.getDefaultJVMPath(), classpath=[jar_path, dependency_path], )
        from satd_detector.core.utils import SATDDetector
        detector1 = SATDDetector()
        for index in range(dataset.num_rows):
            label_pred = 'yes' if detector1.isSATD(dataset['text'][index]) else 'no'
            label_predictions.append(self.format_label(label_pred))

        return super().predict_end(dataset, label_predictions)



# Detect with SATD Text Mining Based SATD Detector

In [95]:

text_mining_based_model = TextMiningBasedSatdDetectorModel('detect', 'text-minig-based-satd-detector', {'yes', 'no'},
                                                           'no')
text_mining_based_model.fit(detect_train_balanced_dataset)
text_mining_based_model.predict(detect_test_dataset)

detect with text-minig-based-satd-detector


java.lang.reflect.InaccessibleObjectException: Unable to make protected final java.lang.Class java.lang.ClassLoader.defineClass(java.lang.String,byte[],int,int,java.security.ProtectionDomain) throws java.lang.ClassFormatError accessible: module java.base does not "opens java.lang" to unnamed module @76a4d6c
	at java.base/java.lang.reflect.AccessibleObject.throwInaccessibleObjectException(AccessibleObject.java:391)
	at java.base/java.lang.reflect.AccessibleObject.checkCanSetAccessible(AccessibleObject.java:367)
	at java.base/java.lang.reflect.AccessibleObject.checkCanSetAccessible(AccessibleObject.java:315)
	at java.base/java.lang.reflect.Method.checkCanSetAccessible(Method.java:203)
	at java.base/java.lang.reflect.Method.setAccessible(Method.java:197)
	at weka.core.WekaPackageClassLoaderManager.injectClasses(WekaPackageClassLoaderManager.java:790)
	at weka.core.WekaPackageClassLoaderManager.injectAllClassesInFromStream(WekaPackageClassLoaderManager.java:688)
	at weka.core.WekaPackageCl

              precision    recall  f1-score   support

          no      0.994     0.991     0.993     12655
         yes      0.667     0.740     0.701       292

    accuracy                          0.986     12947
   macro avg      0.830     0.866     0.847     12947
weighted avg      0.987     0.986     0.986     12947

Merging Mismatch


'./cache/March 23, 2025, 03:58:06$detect_text-minig-based-satd-detector.csv'

In [85]:
import ast
from util import sha1


@retry(
    stop=stop_after_attempt(10),  # Stop after 5 retries
    wait=wait_exponential(multiplier=2, min=60, max=2 * 60),
    retry=retry_if_exception_type(google.api_core.exceptions.ResourceExhausted))
def create_embedding(uri, text):
    return genai.embed_content(
        model=uri,
        content=text,
        task_type="classification")


class GeminiSentenceTransformer:
    def __init__(self, uri: str, use_cache=False):
        self.uri = uri
        self.use_cache = use_cache
        self.file = f'./cache/{uri.split("/")[-1]}.csv'
        if not os.path.exists(self.file):
            with open(self.file, "w") as file:
                file.write("text,hash,embedding")
                file.flush()
        self.cache_df = pd.read_csv(self.file)
        self.encoding_map = {r['hash']: np.array(ast.literal_eval(r['embedding']), dtype=np.float32) for row_index, r in
                             self.cache_df.iterrows()}
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

    def encode(self, features):
        encoded_features = []
        rows = []
        try:
            for text in features:
                if self.use_cache:
                    hash = sha1(text)
                    # embedding_df = self.cache_df[self.cache_df['hash'] == hash]['embedding']
                    if hash not in self.encoding_map:
                        # if embedding_df.empty:
                        response = create_embedding(self.uri, text)
                        embedding_value = response["embedding"]
                        rows.append([text, hash, embedding_value])
                        # self.cache_df.loc[len(self.cache_df)] = [text, hash, embedding_value]
                        # self.cache_df.to_csv(self.file, index=False)
                        # self.cache_df = pd.read_csv(self.file)
                        self.encoding_map[hash] = np.array(embedding_value)
                    encoded_features.append(self.encoding_map[hash])

                    # else:
                    # encoded_features.append(np.array(ast.literal_eval(embedding_df.iloc[0]), dtype=np.float32))
                else:
                    raise Exception('Not Implemented Yet')
        except Exception as e:
            raise e
        finally:
            self.checkpoint(rows)
        return encoded_features

    def checkpoint(self, rows):
        if len(rows) > 0:
            new_df = pd.DataFrame(rows, columns=self.cache_df.columns)
            pd.concat([self.cache_df, new_df]).to_csv(self.file, index=False)
            self.cache_df = pd.read_csv(self.file)


class GeminiEmbeddedLogisticsRegressionModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str,
                 verbose: bool = False):
        super().__init__(task_type, model_uri, known_labels, unmatched_label, verbose)
        self.transformer = GeminiSentenceTransformer(model_uri, True)
        self.model = LogisticRegression()

    def fit(self, dataset: Dataset):
        self.model.fit(self.transformer.encode(dataset['text']), dataset['label'])

    def predict(self, dataset: Dataset):
        super().predict_start(dataset)
        label_predictions = self.model.predict(self.transformer.encode(dataset['text']))
        for index in range(dataset.num_rows):
            label_predictions[index] = self.format_label(label_predictions[index])
        return super().predict_end(dataset, label_predictions)


# Detect with `text-embedding-004` Embedding and Logistic Regression

In [89]:
gemini_embedded_004_model = GeminiEmbeddedLogisticsRegressionModel('detect', 'models/text-embedding-004', {'yes', 'no'},
                                                                   'no')
gemini_embedded_004_model.fit(detect_train_balanced_dataset)
gemini_embedded_004_model.predict(Dataset.from_pandas(detect_test_df[:2000]))

detect with text-embedding-004
              precision    recall  f1-score   support

          no      0.995     0.934     0.963      1949
         yes      0.243     0.804     0.373        51

    accuracy                          0.931      2000
   macro avg      0.619     0.869     0.668      2000
weighted avg      0.975     0.931     0.948      2000

Merging Mismatch


'./cache/March 23, 2025, 03:35:31$detect_text-embedding-004.csv'

# Detect with `text-embedding-004` Embedding including prompt and Logistic Regression

In [70]:
# template = PromptTemplate(
#     name="Manually Crafted",
#     definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
#     instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
#     n_shot_template="Comment: {comment}\nLabel: {label}",
#     line_m_before=3,
#     line_n_after=10
# )
# train_df = detect_train_balanced_df.copy()
# test_df = detect_test_df[:2000].copy()
# for df in [train_df, test_df]:
#     df['text'] = df['text'].apply(lambda text: create_prompt(template, train_dataset, n_shot_idexes, test_dataset,
#                                                              text))
# model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression",
#                            uri='models/text-embedding-004')
# detect_satd('detect', model_config, 0, template, TrainStrategy.N_SHOT_TOP,
#             DatasetDict({'train': Dataset.from_pandas(train_df), 'test': Dataset.from_pandas(test_df)}))

Running text-embedding-004
              precision    recall  f1-score   support

          no      0.993     0.992     0.993      1949
         yes      0.717     0.745     0.731        51

    accuracy                          0.986      2000
   macro avg      0.855     0.869     0.862      2000
weighted avg      0.986     0.986     0.986      2000



# Detect with `models/gemini-embedding-exp-03-07` Embedding and Logistic Regression

In [91]:
gemini_embedded_exp_model = GeminiEmbeddedLogisticsRegressionModel('detect', 'models/gemini-embedding-exp-03-07',
                                                                   {'yes', 'no'}, 'no')
gemini_embedded_exp_model.fit(detect_train_balanced_dataset)
gemini_embedded_exp_model.predict(Dataset.from_pandas(detect_test_df[:2000]))

detect with gemini-embedding-exp-03-07
              precision    recall  f1-score   support

          no      0.996     0.988     0.992      1949
         yes      0.657     0.863     0.746        51

    accuracy                          0.985      2000
   macro avg      0.827     0.925     0.869      2000
weighted avg      0.988     0.985     0.986      2000

Merging Mismatch


'./cache/March 23, 2025, 03:38:26$detect_gemini-embedding-exp-03-07.csv'

# Detect with `models/gemini-embedding-exp-03-07` Embedding including Prompt and Logistic Regression


In [30]:
# model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression",
#                            uri='models/gemini-embedding-exp-03-07')
# template = PromptTemplate(
#     name="Manually Crafted",
#     definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
#     instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
#     n_shot_template="Comment: {comment}\nLabel: {label}",
#     line_m_before=3,
#     line_n_after=10
# )
# train_df = detect_train_balanced_df.copy()
# test_df = detect_test_df[:200].copy()
# for df in [train_df, test_df]:
#     df['text'] = df['text'].apply(lambda text: create_prompt(template, train_dataset, n_shot_idexes, test_dataset,
#                                                              text))
# detect_satd('detect', model_config, 0, template, TrainStrategy.N_SHOT_TOP,
#             DatasetDict({'train': Dataset.from_pandas(train_df), 'test': Dataset.from_pandas(test_df)}))


# Classify with `all-MiniLM-L6-v2` Embedding and Logistic Regression

In [ ]:
detect_satd('classify', MODEL_CONFIG_SENTENCE_EMBEDDED_LR, 0, template, TrainStrategy.N_SHOT_TOP,
            DatasetDict({'train': classify_train_dataset, 'test': classify_test_dataset}))

# Classify with `models/gemini-embedding-exp-03-07` Embedding and Logistic Regression

In [31]:
# model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression",
#                            uri='models/gemini-embedding-exp-03-07')
# detect_satd('classify', model_config, 0, template, TrainStrategy.N_SHOT_TOP,
#             DatasetDict({'train': classify_train_dataset, 'test': classify_test_dataset}))

# Classify with `models/text-embedding-004` Embedding and Logistic Regression

In [ ]:
# model_config = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression",
#                            uri='models/text-embedding-004')
# detect_satd('classify', model_config, 0, template, TrainStrategy.N_SHOT_TOP,
#             DatasetDict({'train': classify_train_dataset, 'test': classify_test_dataset}))

# Classify with Gemini Flash 2.0 N-Shots


In [29]:
# template = PromptTemplate(
#     name="Manually Crafted Classification",
#     definition="You are an AI model trained to classify Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises.",
#     instruction="""Classify Self-Admitted Technical Debt (SATD) in test code into one of the following 16 categories:
#         build : Refers to build issues that make tasks harder and more time-consuming. A project’s build process can include unnecessary code or poorly defined dependencies, causing it to run slowly. This is known as build debt.
#         code: Refers to poor coding practices, such as bad naming conventions, duplicated code and the use of inefficient algorithms which can negatively impact code readability, maintainability, and performance.
#         design: Refers to practices that violate the principles of good object-oriented design, such as high coupling and low cohesion, which can lead to reduced code modularity, flexibility, and maintainability.
#         defect: Unresolved known defects that have been identified and require correction.
#         skip test: Skipping or disabling tests for any reason.
#         temporary fix: A temporary workaround that is intended to address an issue but needs to be replaced with a permanent solution to ensure long-term stability and effectiveness.
#         documentation: Documentation Debt refers to the lack, incompleteness, or outdated state of documentation in a project.
#         impractical case: Refers to the identification of unexpected or problematic situations that are not anticipated to occur under normal circumstances.
#         dependency: Occurs when there is an issue with a dependency.
#         superficial: Occurs when a test only covers partial testing.
#         how to: Occurs when the solution to a problem is unknown or there is uncertainty about how to proceed, often requiring further investigation or exploration to find a resolution.
#         refactor: Refers to debt that mentions about refactoring current code or copying code.
#         requirement: Refers to incomplete or insufficient implementation of test or functionality.
#         multi: Refers to more debt that contain more than one type of debt.
#         other: Any other SATD comments that does not fit into the above classes.
#         """,
#     n_shot_template="<EXAMPLE>\nComment: {comment}\nLabel: {label}\n</EXAMPLE>",
#     line_m_before=3,
#     line_n_after=10
# )
#
# detect_satd('detect', MODEL_CONFIG_GEMINI_2_FLASH, 4, template, TrainStrategy.N_SHOT_TOP,
#             DatasetDict({'train': detect_n_shot_dataset, 'test': Dataset.from_pandas(detect_test_df[:2000])}))

# Merge Result

In [ ]:
# from comment import CommentRepository
# from db_config import SessionLocal
# from dotenv import load_dotenv
# import os
#
# load_dotenv()
# session = SessionLocal()
# repo = CommentRepository()
# comments = repo.get_comments_with_no_prediction(limit=500)
# ids = []
# texts = []
# labels = []
# for comment in comments:
#     ids.append(comment.id)
#     texts.append(comment.text)
#     labels.append(comment.is_td)
#
# test_dataset = Dataset.from_dict({'text': texts, 'label': labels})
# detection_dataset = DatasetDict({
#     'train': dataset['train'],
#     'test': test_dataset
# })
# # print(detection_dataset)
#
# output = detect_satd(MODEL_CONFIGS[0:1], [3], PROMPT_TEMPLATES[0:1], [FewShotSelectionStrategy.MANUAL_CRAFTED], detection_dataset)
# rc, test_x, test_y, pred_y, unknown_labels = output[0]
# for _,[id, pred] in enumerate(zip(ids,  pred_y)):
#     target_comment = repo.get_comment(id)
#     target_comment.pred_td = True if pred.lower() == 'yes' else False
#     session.merge(target_comment)
#     session.commit()


In [29]:

# target_model = 'gemini'
# comment_df = pd.read_csv('../data/comments.csv')
# test_df = comment_df[comment_df[target_model] is None].sample(frac=1, random_state=42).head(10)
#
# ids = []
# texts = []
# labels = []
# for index, row in df.iterrows():
#     ids.append(row['id'])
#     texts.append(row['text'])
#     labels.append(row[target_model])
#
# test_dataset = Dataset.from_dict({'text': texts, 'label': labels})
# detection_dataset = DatasetDict({
#     'train': dataset['train'],
#     'test': test_dataset
# })
#
# output = detect_satd(MODEL_CONFIGS[0:1], [20], PROMPT_TEMPLATES[0:1], [FewShotSelectionStrategy.MANUAL_CRAFTED],
#                      detection_dataset)
# rc, test_x, test_y, pred_y, unknown_labels = output[0]
# for _, [id, pred] in enumerate(zip(ids, pred_y)):
#     comment_df.loc[df['id'] == id, target_model] = pred
# comment_df.to_csv('../data/comments.csv')
#
